# 01 · Dataset access and preprocessing
Public source, strict parsing, label normalization, audits and grouped splits. Execute with the HeroVired (.venv) kernel.

In [1]:
from pathlib import Path
import sys, json
ROOT=Path.cwd() if (Path.cwd()/'src').exists() else Path.cwd().parent
sys.path.insert(0,str(ROOT/'src'))
import pandas as pd
import numpy as np
def read_json(path): return json.loads((ROOT/path).read_text())

In [2]:
from resume_matcher.data import DATASET_ID, DATASET_REVISION, load_huggingface_dataset
print(DATASET_ID, DATASET_REVISION)
# Optional network refresh: raw = load_huggingface_dataset(split='train')
raw=pd.read_csv(ROOT/'data/raw/original_train.csv')
raw.shape, raw.columns.tolist()

facehuggerapoorv/resume-jd-match bfab7acecac062628b39e0dd0c59e72d675edc9c


((6241, 5), ['candidate_name', 'job_title', 'resume_text', 'jd_text', 'label'])

In [3]:
from resume_matcher.research import audit_frame, clean_training_data, grouped_split
from resume_matcher.preprocessing import clean_text
print(json.dumps(audit_frame(raw),indent=2))
clean, quarantine=clean_training_data(raw)
train, validation=grouped_split(clean)
print({'clean_pairs':len(clean),'quarantined_rows':len(quarantine),'train':len(train),'validation':len(validation)})

{
  "rows": 6241,
  "nulls": {
    "candidate_name": 0,
    "job_title": 0,
    "resume_text": 0,
    "jd_text": 0,
    "label": 0
  },
  "empty_text": {
    "resume_text": 0,
    "jd_text": 0
  },
  "labels": {
    "low_fit": 3143,
    "potential_fit": 1556,
    "strong_fit": 1542
  },
  "duplicate_pairs": 7,
  "conflicting_pairs": 6,
  "unique_jds": 280,
  "unique_resumes": 642
}
{'clean_pairs': 6228, 'quarantined_rows': 12, 'train': 5025, 'validation': 1203}


In [4]:
assert not set(train.jd_text.map(clean_text)) & set(validation.jd_text.map(clean_text))
test=pd.read_csv(ROOT/'data/raw/original_test.csv')
assert not set(clean.jd_text.map(clean_text)) & set(test.jd_text.map(clean_text))
print('JD groups are disjoint. Test rows:',len(test))
print('Normalized sample:',clean_text(raw.iloc[0].resume_text)[:250])

JD groups are disjoint. Test rows: 1759
Normalized sample: summaryhighly motivated sales associate with extensive customer service and sales experience. outgoing sales professional with track record of driving increased sales improving buying experience and elevating company profile with target market. highl


Conflicting TRAIN labels never consult test labels. The upstream card is empty, so synthetic-data status and annotation provenance remain uncertain. Reused resumes limit claims about unseen candidates.